# Módulo 1A — Equilibrios y estabilidad local

**Modelo del enunciado (una neurona, forzamiento constante $I$):**

$$\dot v=v-\frac{v^3}{3}-w+I,\qquad \dot w=\frac{v+a-bw}{\tau},\qquad a\ge0,\ 0\le b<1,\ \tau>0.$$

Este notebook: (1) deduce el equilibrio, (2) calcula jacobiano y autovalores, (3) clasifica estabilidad local y (4) dibuja nulclinas, campo vectorial, trayectorias y barrido de parámetros. El control variable en el tiempo se reserva para módulos posteriores.

**Visualización:** los gráficos aparecen dentro de Jupyter y no se guardan en el disco. Más abajo hay parámetros interactivos y una animación reproducible.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from dataclasses import replace

# Ejecuta Jupyter desde la raíz del repositorio o desde notebooks/
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    raise RuntimeError('Abre Jupyter desde la raíz del repositorio o desde notebooks/.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from fhn_control.models import FHNParameters, rhs, equilibrium, jacobian, hopf_candidates
from fhn_control.visualization import plot_phase_portrait, animate_trajectory

## 1. Equilibrios

Se impone $\dot v=\dot w=0$. Eliminando $w$ se obtiene

$$b(v^*)^3+3(1-b)v^*+3a-3bI=0,\qquad w^*=v^*-\frac{(v^*)^3}{3}+I.$$

La derivada del polinomio es $3b(v^*)^2+3(1-b)>0$: **existe una única raíz real** para $0\le b<1$ (también si $b=0$). No dividimos por $b$ para mantener ese caso.

## 2. Estabilidad local

$$J(v^*)=\begin{pmatrix}1-(v^*)^2&-1\\1/\tau&-b/\tau\end{pmatrix},\quad
\mathrm{tr}J=1-(v^*)^2-b/\tau,\quad
\det J=[1-b+b(v^*)^2]/\tau>0.$$

Para $\mathrm{tr}J<0$: equilibrio localmente asintóticamente estable; para $\mathrm{tr}J>0$: inestable. Cuando $\mathrm{tr}J=0$ la linealización es **inconclusa**. El discriminante $(\mathrm{tr}J)^2-4\det J$ distingue foco (negativo) y nodo (positivo).

In [ ]:
p=FHNParameters(a=0.7,b=0.8,tau=12.5,I=0.2)
xstar=equilibrium(p)
J=jacobian(xstar[0],p)
eigs=np.linalg.eigvals(J)
trace,det=np.trace(J),np.linalg.det(J)
delta=trace**2-4*det
print('Parámetros:',p)
print('Equilibrio (v*,w*):',np.round(xstar,6))
print('Norma de f(X*):',np.linalg.norm(rhs(0,xstar,p)))
print('Jacobiano:\n',J)
print('Autovalores:',eigs)
print('Traza, determinante, discriminante:',trace,det,delta)
print('Estabilidad:', 'localmente asintóticamente estable' if trace < -1e-9 else 'inestable' if trace > 1e-9 else 'linealización inconclusa')


## 3. Nulclinas y retrato de fases

La nulclina de $v$ es $w=v-v^3/3+I$; para $b>0$, la de $w$ es $w=(v+a)/b$. Su intersección es el equilibrio. Las curvas muestran la geometría del sistema, y las flechas el sentido del flujo.

In [ ]:
v=np.linspace(-2.5,2.5,33); w=np.linspace(-1.5,2.8,29)
V,W=np.meshgrid(v,w)
D=rhs(0,np.array([V,W]),p)
speed=np.hypot(D[0],D[1]); speed=np.maximum(speed,1e-12)
fig,ax=plt.subplots(figsize=(8,5.8))
ax.quiver(V,W,D[0]/speed,D[1]/speed,alpha=.32,angles='xy')
vv=np.linspace(-2.5,2.5,800)
ax.plot(vv,vv-vv**3/3+p.I,label=r'$\dot v=0$',lw=2)
if p.b>0: ax.plot(vv,(vv+p.a)/p.b,label=r'$\dot w=0$',lw=2)
else: ax.axvline(-p.a,label=r'$\dot w=0$',lw=2)
for x0 in [(-2,-.7),(-1,2),(0,2.4),(1.7,2.2),(2,.2)]:
 sol=solve_ivp(lambda t,x:rhs(t,x,p),(0,150),x0,max_step=.2,rtol=1e-8,atol=1e-10)
 ax.plot(sol.y[0],sol.y[1],lw=1.2,alpha=.82)
ax.scatter(*xstar,s=90,color='black',zorder=6,label='Equilibrio')
ax.set(xlim=(-2.5,2.5),ylim=(-1.5,2.8),xlabel='$v$',ylabel='$w$',title='Nulclinas, flujo y trayectorias')
ax.legend(loc='upper left',fontsize=9);ax.grid(alpha=.18)
plt.show()


## 4. Cambio de estabilidad al variar $I$

Se conserva $(a,b,\tau)$ y se calcula el equilibrio y la traza de su jacobiano para cada forzamiento constante. Las intersecciones con traza cero son **candidatos** a Hopf, no pruebas de existencia de ciclos.

In [ ]:
Is=np.linspace(0,1.8,250)
vs=[];traces=[];reals=[]
for I in Is:
 pp=replace(p,I=float(I)); ve=equilibrium(pp)[0]
 vs.append(ve);traces.append(np.trace(jacobian(ve,pp)))
 reals.append(max(np.linalg.eigvals(jacobian(ve,pp)).real))
candidates=hopf_candidates(p)
fig,axs=plt.subplots(1,2,figsize=(11,4))
axs[0].plot(Is,vs);axs[0].set(xlabel='$I$',ylabel='$v^*$',title='Equilibrio según el forzamiento')
axs[1].plot(Is,traces,label='Traza');axs[1].axhline(0,color='black',lw=1)
for vcrit,Icrit in candidates:
 for ax in axs: ax.axvline(Icrit,color='tab:red',ls='--',alpha=.75)
axs[1].set(xlabel='$I$',ylabel='traza del jacobiano',title='Cambio de estabilidad')
for ax in axs: ax.grid(alpha=.2)
fig.tight_layout();plt.show()
print('Candidatos críticos (v*, I*):', candidates)


## Conclusión

La estabilidad determinada por el jacobiano es **local** y vale cuando los autovalores son hiperbólicos. Los valores críticos identifican posibles transiciones oscilatorias. El siguiente notebook investiga periódicos, condiciones de Hopf y amplitudes numéricas.

## 5. Experimento interactivo: ¿cómo evoluciona una trayectoria cerca de un equilibrio?

Cambia el forzamiento constante $I$ y las perturbaciones $(\delta v,\delta w)$ respecto del equilibrio. El estado inicial es $X(0)=X^*+(\delta v,\delta w)$. El plano de fases muestra la trayectoria y las nulclinas; a la derecha se dibuja la evolución temporal. **La clasificación mediante autovalores es local**, y el caso de traza nula requiere análisis no lineal.

Los deslizadores actualizan figuras *dentro* del notebook; no se generan archivos. El cálculo se efectúa al soltar cada deslizador.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

base = FHNParameters(a=0.7, b=0.8, tau=12.5, I=0.2)
I_slider = widgets.FloatSlider(value=0.2,min=0,max=1.8,step=0.05,description='I',continuous_update=False)
dv_slider = widgets.FloatSlider(value=0.15,min=-1.0,max=1.0,step=0.05,description='delta v',continuous_update=False)
dw_slider = widgets.FloatSlider(value=0.08,min=-1.0,max=1.0,step=0.05,description='delta w',continuous_update=False)
T_slider = widgets.IntSlider(value=110,min=30,max=350,step=10,description='Tiempo',continuous_update=False)

def dibujar_equilibrio(I, dv, dw, T):
    pp = replace(base, I=I)
    eq = equilibrium(pp)
    x0 = eq + [dv, dw]
    sol = solve_ivp(lambda t, x: rhs(t,x,pp), (0,T), x0,
                    t_eval=np.linspace(0,T,1100),rtol=1e-8,atol=1e-10)
    J = jacobian(eq[0],pp)
    eig = np.linalg.eigvals(J)
    tr = np.trace(J)
    condition = ('atractor local' if tr < -1e-8 else
                 'inestable' if tr > 1e-8 else 'inconcluso por linealización')
    print(f'Equilibrio={np.round(eq,4)}, autovalores={np.round(eig,4)}; {condition}')
    fig, axs = plt.subplots(1,2,figsize=(11,4.5))
    plot_phase_portrait(axs[0],pp,sol=sol, show_nullclines=True,
                        xlim=(-2.7,2.7),ylim=(-1.7,2.9))
    axs[0].set_title('Espacio de fases (v,w)')
    axs[1].plot(sol.t,sol.y[0],label='v(t)')
    axs[1].plot(sol.t,sol.y[1],label='w(t)')
    axs[1].axhline(eq[0],ls='--',alpha=0.5,color='C0')
    axs[1].axhline(eq[1],ls='--',alpha=0.5,color='C1')
    axs[1].set(xlabel='t',title='Evolución temporal')
    axs[1].legend();axs[1].grid(alpha=.2)
    fig.tight_layout();plt.show()

salida_eq = widgets.interactive_output(dibujar_equilibrio, {'I':I_slider,
              'dv':dv_slider,'dw':dw_slider,'T':T_slider})
display(widgets.VBox([widgets.HBox([I_slider,T_slider]),
                      widgets.HBox([dv_slider,dw_slider])]), salida_eq)

## 6. Animación de una trayectoria alrededor del equilibrio

Presiona **Generar / reproducir** para ver un punto que recorre la solución de la EDO. La trayectoria se va dibujando a medida que transcurre el tiempo. La animación aparece en Jupyter con controles de reproducción, pausa y avance manual. Toma los parámetros que hayas elegido arriba.

La animación se genera bajo demanda para que la ejecución de todo el notebook no sea lenta. No se exporta ningún GIF, MP4 ni PNG.

In [ ]:
boton_eq = widgets.Button(description='Generar / reproducir',button_style='info',icon='play')
salida_anim_eq = widgets.Output()

def reproducir_eq(_):
    with salida_anim_eq:
        clear_output(wait=True)
        pp = replace(base,I=I_slider.value)
        eq = equilibrium(pp)
        x0 = eq + np.array([dv_slider.value,dw_slider.value])
        T = T_slider.value
        ts = np.linspace(0,T,600)
        sol = solve_ivp(lambda t,x:rhs(t,x,pp), (0,T),x0,t_eval=ts,
                        rtol=1e-8,atol=1e-10)
        anim = animate_trajectory(sol,pp,max_frames=105,title='Evolución alrededor del equilibrio')
        display(HTML(anim.to_jshtml(fps=20,default_mode='once')))
        plt.close(anim._fig)

boton_eq.on_click(reproducir_eq)
display(boton_eq,salida_anim_eq)